# Ensemble Models for Credit Card Fraud Detection

Second part of the fraud-detection project. `fraud_detection.ipynb` showed that a single decision tree tops out around 0.75 F1 on the fraud class and that oversampling does not help much. This notebook asks whether ensembles do better, comparing four families on the same split: **voting, bagging (including random forests), boosting, and stacking**. Every model is scored on the fraud class by precision, recall, and F1, with the confusion matrix printed so false positives and missed frauds are visible directly.

## Preprocessing

Preprocessing follows `fraud_detection.ipynb`: transaction time is converted to hour of day and standardized, amount is scaled with `RobustScaler`, and the data is split 60/40 so the test set keeps 199 fraud cases.


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler

## [ Loading Dataset ]
print("Loading Kaggle credit card transactions dataset...")
df = pd.read_csv("creditcard.csv")

## [ Feature Scaling ]
print("Scaling transaction time and amount features...")
# Convert transaction time from seconds to hours in a day
df["Time"] = (df["Time"]/(60*60))%24
# Scale time features with StandardScaler (suitable for normally distributed data)
df["Time"] = StandardScaler().fit_transform(df["Time"].values[:, None])
# Scale amount with RobustScaler (robust to outliers, useful for skewed distributions)
df["Amount"] = RobustScaler().fit_transform(df["Amount"].values[:, None])

## [ Feature-label / train-test splits ]
print("Performing feature-label / train-test splits...")

# Get feature and label values from original dataset
feat_all = df.drop(["Class"], axis=1).values
y_all = df["Class"].values

# Split samples into training and test sets
feat_train, feat_test, y_train, y_test = train_test_split(
    feat_all, y_all, test_size=0.4, random_state=0
)

print("Completed.")

Loading Kaggle credit card transactions dataset...
Scaling transaction time and amount features...
Performing feature-label / train-test splits...
Completed.


Two helpers from `viz_util.py` are used throughout: `timeit` times the block it wraps, and `evaluate_model` prints the classification report and confusion matrix for a fitted model on the test set.

## Baseline: logistic regression


In [3]:
from sklearn.linear_model import LogisticRegression

from viz_util import timeit, evaluate_model

# Time the training of logistic regression classifier
with timeit("Training logistic regression classifier"):
    logistic_model = LogisticRegression(max_iter=200).fit(feat_train, y_train)

# Evaluate trained model and print metrics
evaluate_model(logistic_model, "logistic regression classifier", feat_test, y_test)

Training logistic regression classifier started...
Training logistic regression classifier completed. Elapsed time: 1.74s

[ Evaluation result for logistic regression classifier ]
Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    113724
           1       0.89      0.59      0.71       199

    accuracy                           1.00    113923
   macro avg       0.94      0.80      0.86    113923
weighted avg       1.00      1.00      1.00    113923

Confusion matrix:
[[113709     15]
 [    81    118]] 



## Voting

A voting ensemble trains several different classifiers independently and combines their predictions. With **hard voting** the ensemble label is the majority class across models. With **soft voting** the class probabilities are averaged and the highest-probability class wins, which also avoids ties.

The ensemble below combines logistic regression, Gaussian naive Bayes, and a decision tree. Each sub-classifier is also evaluated on its own so the gain from voting is visible.


In [4]:
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import VotingClassifier

# Hard-voting ensemble
voting_ensemble = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(max_iter=200)),
        ('nb', GaussianNB()),
        ('dt', DecisionTreeClassifier())
    ],
    voting='hard'
)

with timeit("Training voting ensemble (hard voting)"):
    voting_ensemble.fit(feat_train, y_train)

# Evaluate the voting ensemble classifier
evaluate_model(voting_ensemble, "voting ensemble (hard voting)", feat_test, y_test)

# Each sub-classifier on its own
for name, sub_model in voting_ensemble.named_estimators_.items():
    evaluate_model(sub_model, f"sub-classifier: {name}", feat_test, y_test)

# Same ensemble with soft voting
voting_ensemble_soft = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(max_iter=200)),
        ('nb', GaussianNB()),
        ('dt', DecisionTreeClassifier())
    ],
    voting='soft'
)

with timeit("Training voting ensemble (soft voting)"):
    voting_ensemble_soft.fit(feat_train, y_train)

evaluate_model(voting_ensemble_soft, "voting ensemble (soft voting)", feat_test, y_test)

Training voting ensemble (hard voting) started...
Training voting ensemble (hard voting) completed. Elapsed time: 11.21s

[ Evaluation result for voting ensemble (hard voting) ]
Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    113724
           1       0.79      0.76      0.78       199

    accuracy                           1.00    113923
   macro avg       0.89      0.88      0.89    113923
weighted avg       1.00      1.00      1.00    113923

Confusion matrix:
[[113683     41]
 [    47    152]] 

[ Evaluation result for sub-classifier: lr ]
Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    113724
           1       0.89      0.59      0.71       199

    accuracy                           1.00    113923
   macro avg       0.94      0.80      0.86    113923
weighted avg       1.00      1.00      1.00    113923

Confusion matrix:
[[11

## Bagging and random forests

A **bagging** ensemble trains many copies of the same model, each on a random subset of samples (and optionally features), and votes among them. It reduces the variance of high-capacity models, so it works best with deep trees or neural networks and is expected to do little for a linear model.

To check that, the cell below bags logistic regression under four settings: number of sub-classifiers (10 vs. 30), sample fraction (20% vs. 60%), and feature subsampling (50% of features).


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import BaggingClassifier

# Four bagging settings for logistic regression
bagging_models = {
    "10 classifiers, 20% samples": BaggingClassifier(
        estimator=LogisticRegression(max_iter=200),
        n_estimators=10,
        max_samples=0.2,
        random_state=0
    ),
    "30 classifiers, 20% samples": BaggingClassifier(
        estimator=LogisticRegression(max_iter=200),
        n_estimators=30,
        max_samples=0.2,
        random_state=0
    ),
    "10 classifiers, 60% samples": BaggingClassifier(
        estimator=LogisticRegression(max_iter=200),
        n_estimators=10,
        max_samples=0.6,
        random_state=0
    ),
    "10 classifiers, 20% samples, 50% features": BaggingClassifier(
        estimator=LogisticRegression(max_iter=200),
        n_estimators=10,
        max_samples=0.2,
        bootstrap_features=True,
        max_features=0.5,
        random_state=0
    ),
}

for setting, model in bagging_models.items():
    # Train each bagging classifier
    with timeit(f"Training bagging ensemble ({setting})"):
        model.fit(feat_train, y_train)
    # Evaluate each bagging classifier
    evaluate_model(model, f"bagging ensemble ({setting})", feat_test, y_test)

Training bagging ensemble (10 classifiers, 20% samples) started...
Training bagging ensemble (10 classifiers, 20% samples) completed. Elapsed time: 15.61s

[ Evaluation result for bagging ensemble (10 classifiers, 20% samples) ]
Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    113724
           1       0.89      0.57      0.70       199

    accuracy                           1.00    113923
   macro avg       0.94      0.79      0.85    113923
weighted avg       1.00      1.00      1.00    113923

Confusion matrix:
[[113710     14]
 [    85    114]] 

Training bagging ensemble (30 classifiers, 20% samples) started...
Training bagging ensemble (30 classifiers, 20% samples) completed. Elapsed time: 44.78s

[ Evaluation result for bagging ensemble (30 classifiers, 20% samples) ]
Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    113724
     

A **random forest** is bagging over decision trees where each tree sees both a subset of samples and a subset of features at every split. Below, a single decision tree is compared with random forests of 40 and 100 trees.


In [6]:
import os
from sklearn.ensemble import RandomForestClassifier

# Number of CPUs for ensemble learning methods
N_ENSEMBLE_CPUS = max(os.cpu_count()//2, 1)

# A regular decision tree classifier
with timeit("Training DT classifier"):
    dt_model = DecisionTreeClassifier()
    dt_model.fit(feat_train, y_train)

# Random forest with 40 trees
with timeit("Training RF classifier (40 DTs)"):
    rf_40_model = RandomForestClassifier(n_estimators=40, n_jobs=N_ENSEMBLE_CPUS, random_state=0)
    rf_40_model.fit(feat_train, y_train)

# Random forest with 100 trees
with timeit("Training RF classifier (100 DTs)"):
    rf_100_model = RandomForestClassifier(n_estimators=100, n_jobs=N_ENSEMBLE_CPUS, random_state=0)
    rf_100_model.fit(feat_train, y_train)

# Evaluate previous models
evaluate_model(dt_model, "DT classifier", feat_test, y_test)
evaluate_model(rf_40_model, "Random forest classifier (40 DTs)", feat_test, y_test)
evaluate_model(rf_100_model, "Random forest classifier (100 DTs)", feat_test, y_test)

Training DT classifier started...
Training DT classifier completed. Elapsed time: 8.36s

Training RF classifier (40 DTs) started...
Training RF classifier (40 DTs) completed. Elapsed time: 12.08s

Training RF classifier (100 DTs) started...
Training RF classifier (100 DTs) completed. Elapsed time: 30.54s

[ Evaluation result for DT classifier ]
Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    113724
           1       0.75      0.74      0.75       199

    accuracy                           1.00    113923
   macro avg       0.88      0.87      0.87    113923
weighted avg       1.00      1.00      1.00    113923

Confusion matrix:
[[113676     48]
 [    52    147]] 

[ Evaluation result for Random forest classifier (40 DTs) ]
Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    113724
           1       0.94      0.76      0.84       199

 

## Boosting

Boosting builds a strong classifier by adding weak learners one at a time, re-weighting the training samples after each round so the next learner focuses on what the previous ones got wrong. It reduces bias as well as variance.

Two variants are tried, both using decision stumps as the weak learner: [AdaBoost](https://en.wikipedia.org/wiki/AdaBoost), which re-weights samples by classification error, and [gradient boosting](https://en.wikipedia.org/wiki/Gradient_boosting), which fits each new stump to the gradient of the loss.


In [7]:
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier

# AdaBoost: adjusts weights based on misclassification rates (50 default weak learners)
with timeit("Training AdaBoost classifier (50 DTs)"):
    adaboost_model = AdaBoostClassifier()
    adaboost_model.fit(feat_train, y_train)

# Gradient boosting: sequentially corrects errors using gradient descent (40 estimators)
with timeit("Training gradient boosting classifier"):
    gb_model = GradientBoostingClassifier(n_estimators=40)
    gb_model.fit(feat_train, y_train)

# Evaluate boosting models
evaluate_model(adaboost_model, "AdaBoost classifier", feat_test, y_test)
evaluate_model(gb_model, "gradient boosting classifier", feat_test, y_test)

Training AdaBoost classifier (50 DTs) started...
Training AdaBoost classifier (50 DTs) completed. Elapsed time: 28.91s

Training gradient boosting classifier started...
Training gradient boosting classifier completed. Elapsed time: 63.72s

[ Evaluation result for AdaBoost classifier ]
Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    113724
           1       0.78      0.66      0.72       199

    accuracy                           1.00    113923
   macro avg       0.89      0.83      0.86    113923
weighted avg       1.00      1.00      1.00    113923

Confusion matrix:
[[113687     37]
 [    67    132]] 

[ Evaluation result for gradient boosting classifier ]
Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    113724
           1       0.89      0.60      0.72       199

    accuracy                           1.00    113923
   macro avg 

## Stacking

Stacking trains several base classifiers with $K$-fold cross-validation, collects their out-of-fold predictions as features, and fits a **meta-classifier** on those. At prediction time each base classifier's fold clones are averaged, the outputs are concatenated, and the meta-classifier produces the final label.

The stack below uses a random forest, logistic regression, and a linear SVM as base models with a logistic-regression meta-classifier.


In [8]:
from sklearn.svm import LinearSVC
from sklearn.ensemble import StackingClassifier

with timeit("Training stacking ensemble"):
    # Create a stacking ensemble with a logistic regression meta-classifier and three sub-classifiers
    stacking_ensemble = StackingClassifier([
        ("Random forest", RandomForestClassifier(n_estimators=40)),
        ("Logistic", LogisticRegression(max_iter=200)),
        ("SVM", LinearSVC(max_iter=1500))
    ], LogisticRegression(), n_jobs=N_ENSEMBLE_CPUS)
    # Train the stacking ensemble
    stacking_ensemble.fit(feat_train, y_train)

# Evaluate the stacking ensemble
evaluate_model(stacking_ensemble, "stacking ensemble", feat_test, y_test)

Training stacking ensemble started...


Training stacking ensemble completed. Elapsed time: 136.89s

[ Evaluation result for stacking ensemble ]
Classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00    113724
           1       0.95      0.66      0.78       199

    accuracy                           1.00    113923
   macro avg       0.97      0.83      0.89    113923
weighted avg       1.00      1.00      1.00    113923

Confusion matrix:
[[113717      7]
 [    68    131]] 



## Observations

- **Random forests are the clear winner**: 0.84 F1 on the fraud class with only 9-10 false positives out of 113,724 legitimate transactions, versus 0.75 for a single tree. Going from 40 to 100 trees changes nothing measurable.
- **Voting helps weak, diverse models.** Hard and soft voting over LR, naive Bayes, and a tree reach 0.78 F1, above any of the three alone. Naive Bayes on its own is unusable here (0.06 precision), but it still contributes recall to the vote.
- **Bagging a linear model does nothing.** All four bagged-logistic-regression settings land at 0.66-0.70 F1, no better than plain logistic regression (0.71). Bagging needs a high-variance base model to have something to average out.
- **Boosting with stumps underperforms** the forest (0.72 F1 for both AdaBoost and gradient boosting), and gradient boosting is the slowest model here.
- **Stacking gives the highest precision** (0.95, 7 false positives) but misses more fraud than the forest (recall 0.66 vs. 0.76), for 0.78 F1. It is also the most expensive to train.

## References

1. Ensemble learning: https://en.wikipedia.org/wiki/Ensemble_learning
2. Random forest: https://en.wikipedia.org/wiki/Random_forest
3. Boosting: https://en.wikipedia.org/wiki/Boosting_(machine_learning)
4. AdaBoost: https://en.wikipedia.org/wiki/AdaBoost
5. Gradient boosting: https://en.wikipedia.org/wiki/Gradient_boosting
